For ticket https://github.com/biglocalnews/bln-planning-eng/issues/304

Notes:

- looks like legistar scraper requires an http schema - so i added `https://` to the start of the urls I tested
- some sites still fail - in ways that seem like the specific sites are likely on an unsupported version of Legistar

What to do?

- I'd suggest trying to scrape all the sites (add in the schema before scraping!)
- For those that fail - please just mark as 'no' and capture whatever the error message is. (A try and except catch would be a great option for this!) This way we can see if there are trends in how sites fail - eg if a lot fail on `KeyError: 'Meeting Details'`

In [ ]:
!pip install civic-scraper==1.0.0rc2.dev0 pendulum

In [ ]:
from importlib import metadata
from civic_scraper.base.cache import Cache
from civic_scraper.platforms import CivicPlusSite, LegistarSite, DigitalTowPathSite
from requests.exceptions import SSLError, ConnectionError, Timeout


CIVICPLUS = "civicplus"
LEGISTAR = "legistar"
DIGITAL_TOW_PATH = "digital_tow_path"

SUPPORTED = {CIVICPLUS, LEGISTAR, DIGITAL_TOW_PATH}


def scrape_agency_civic_scraper(
    cache_dir: str, url: str, metadata: dict, start: str, end: str
) -> list[dict]:
    """
    Scrapes metadata for documents from a civic scraper
    site within a given date range.

    Initializes a scraper using the given `url` and associated metadata
    (e.g., place name). Uses a local cache directory and retrieves assets between the specified
    `start` and `end` dates.

    Args:
        cache_dir (str): Path to the local cache directory.
        url (str): URL to scrape.
        metadata (dict): metadata for url
        start (str): Start date in YYYY-MM-DD format.
        end (str): End date in YYYY-MM-DD format.

    Returns:
        list: A list of asset metadata dictionaries retrieved from the site.

    Raises:
        Exception: Re-raises any error encountered during scraping after logging a stack trace.
    """
    assets_meta = []
    try:
        print(f"Scraping {url} from {start} to {end}")
        site_type = metadata.get("site_type")

        if site_type not in SUPPORTED:
            print(f"Invalid 'site_type' in metadata for {url} - got {site_type}")
            return []

        site_builders = {
            CIVICPLUS: lambda: CivicPlusSite(url, cache=Cache(cache_dir), place_name=metadata.get("name")),
            LEGISTAR: lambda: LegistarSite(url, timezone="US/Eastern"),
            DIGITAL_TOW_PATH: lambda: DigitalTowPathSite(url),
        }

        # attempt scrape
        assets_meta = site_builders[site_type]().scrape(start_date=start, end_date=end)

    except (SSLError, ConnectionError, Timeout) as e:
        # Handle SSL or network-related issues gracefully
        print(
            f"⚠️  Skipping {url} due to connection or SSL error: {e.__class__.__name__} - {e}"
        )
        return []

    except Exception as e:
        err_msg = f"ERROR ON SCRAPER TASK for {url}. Here's the stack trace:\n"
        print(err_msg)
        raise e

    return assets_meta



In [ ]:
import pendulum


# Check two months back and into future
START_DATE_DEFAULT = pendulum.today().subtract(months=2).strftime("%Y-%m-%d")
END_DATE_DEFAULT = pendulum.today().add(months=2).strftime("%Y-%m-%d")

metadata = {'site_type': 'legistar'}
start_date = START_DATE_DEFAULT
end_date = END_DATE_DEFAULT
CACHE_DIR_CIVIC_SCRAPER = ""

# NOTE: had to `https://` to the start of these to get the package to recognize them
urls_to_test__WORK = [
"https://newark.legistar.com/Calendar.aspx",
"https://paradisevalleyaz.legistar.com/Calendar.aspx",
"https://yuma-az.legistar.com/Calendar.aspx",
"https://crd.ca.legistar.com/Calendar.aspx"
]

urls_to_test__DONT_WORK = [
    "https://phoenix.legistar.com/Calendar.aspx",
    "https://pima.legistar.com/Calendar.aspx"
]

for url in urls_to_test__WORK + urls_to_test__DONT_WORK:
  assets_metadata_lst = scrape_agency_civic_scraper(CACHE_DIR_CIVIC_SCRAPER, url, metadata, start_date, end_date)

  print(f"Assets found for {url}: {len(assets_metadata_lst)}")


Scraping https://newark.legistar.com/Calendar.aspx from 2026-01-18 to 2026-05-18


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'newark.legistar.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'newark.legistar.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'newark.legistar.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connect

Assets found for https://newark.legistar.com/Calendar.aspx: 8
Scraping https://paradisevalleyaz.legistar.com/Calendar.aspx from 2026-01-18 to 2026-05-18


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paradisevalleyaz.legistar.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paradisevalleyaz.legistar.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paradisevalleyaz.legistar.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Assets found for https://paradisevalleyaz.legistar.com/Calendar.aspx: 15
Scraping https://yuma-az.legistar.com/Calendar.aspx from 2026-01-18 to 2026-05-18


...

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'crd.ca.legistar.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'crd.ca.legistar.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'crd.ca.legistar.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connect

Assets found for https://crd.ca.legistar.com/Calendar.aspx: 24
Scraping https://phoenix.legistar.com/Calendar.aspx from 2026-01-18 to 2026-05-18


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'phoenix.legistar.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'phoenix.legistar.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


ERROR ON SCRAPER TASK for https://phoenix.legistar.com/Calendar.aspx. Here's the stack trace:



KeyError: 'Meeting Details'